![Evidence 3](https://i.imgur.com/mu6ZuGT.jpg)

# Instituto Tecnológico y de Estudios Superiores de Monterrey
## Maestría en Inteligencia Artificial Aplicada
### Proyecto Integrador (Gpo 10)

### **Proyecto: Diseño Acelerado de Fármacos Agonistas de la Proteína PD-L1**

### Avance 3: Baseline

#### **Docentes:**
- Dra. Grettel Barceló Alonso - Profesor Titular
- Dra. Eduviges Ludivina Facundo Flores  – Profesor Tutor

### **Asesores**
- Dr. Juan Arturo Nolazco Flores
- Dr. Carlos Alberto Brizuela Rodríguez

### **Equipo 2:**

* Maritza Liceth Guerrero Millán: A01795976
* Juan Luis García Chávez: A01795943
* Miguel Ángel Loya Tarango: A01796407

## Introducción

En la etapa anterior, la construcción del modelo *baseline* nos permitió confirmar que existe una fuerte señal predictiva en nuestros datos, sin embargo, nos enfrentamos a un problema severo de sobreajuste (overfitting) debido a la escasez de muestras ($N=40$). Para este avance, hemos solucionado esta limitante **expandiendo nuestro dataset a 2080 registros**, lo que nos proporciona una base estadística mucho más robusta.

El objetivo de esta libreta es explorar, evaluar y afinar múltiples enfoques de Machine Learning para predecir la métrica de interacción proteína-proteína (`i_ptm`). 

Cumpliendo con los lineamientos del proyecto, en este avance:
1. **Construiremos al menos 6 modelos predictivos puramente individuales** (sin utilizar algoritmos de ensamble como Random Forest o Gradient Boosting).
2. **Compararemos su rendimiento** utilizando múltiples métricas, incluyendo el Error Absoluto Medio (MAE), la correlación de Spearman, $R^2$, y el tiempo de entrenamiento (Fit Time).
3. **Realizaremos un Ajuste Fino (Fine-Tuning)** sobre los dos mejores modelos encontrados.
4. **Seleccionaremos y justificaremos el modelo individual final** considerando no solo las métricas matemáticas, sino también su interpretabilidad y viabilidad computacional para el negocio farmacéutico.


---
## Comparativa de Modelos Individuales

Para cumplir con el requisito de utilizar **algoritmos variados e individuales (no ensambles)**, hemos seleccionado 8 arquitecturas distintas que abarcan desde métodos lineales simples hasta enfoques no lineales basados en distancias, árboles y redes neuronales. 

Los modelos a evaluar son:
* **DummyRegressor:** Nuestro baseline ingenuo de referencia (predice la media).
* **Ridge, Lasso y ElasticNet:** Modelos lineales con distintos tipos de regularización (L2, L1, e híbrida) para manejar la posible multicolinealidad de las características de secuencia.
* **DecisionTreeRegressor:** Modelo no lineal basado en particiones del espacio de características.
* **KNeighborsRegressor (KNN):** Modelo no lineal basado en similitud de instancias.
* **SVR:** Máquina de Vectores de Soporte con kernel radial (RBF).
* **MLPRegressor:** Perceptrón multicapa (Red Neuronal) para capturar patrones altamente complejos.

Evaluaremos estos modelos usando Validación Cruzada (`KFold` con 5 splits) y consolidaremos los resultados en una tabla que ordenará los algoritmos por nuestra métrica principal (MAE), incluyendo los tiempos de entrenamiento (`Fit_Time_s`).

In [8]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, cross_val_predict, KFold, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

In [9]:

# 1. Cargar el nuevo dataset
df = pd.read_csv('dataset.csv')

# 2. Ingeniería de Características (Extrayendo solo el péptido)
df['peptide'] = df['seq'].str.split('/').str[1]
df['length'] = df['peptide'].str.len()

amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
seq_features = ['length']

for aa in amino_acids:
    col_freq = f"freq_{aa}"
    col_prop = f"prop_{aa}"
    df[col_freq] = df['peptide'].str.count(aa)
    df[col_prop] = df[col_freq] / df['length']
    seq_features.append(col_prop)

# 3. Variables predictoras y Variable Objetivo
target = 'i_ptm'
numeric_cols = ['mpnn', 'plddt', 'ptm', 'pae', 'i_pae', 'rmsd']

X_all = df[seq_features + numeric_cols]
y = df[target]

print("Shape de X_all:", X_all.shape)

# 4. Definir 8 modelos estrictamente INDIVIDUALES (No ensambles)
models = {
    "Dummy (Baseline)": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(alpha=0.001, random_state=42),
    "ElasticNet": ElasticNet(alpha=0.01, random_state=42),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR (RBF)": SVR(),
    "MLP (Neural Net)": MLPRegressor(hidden_layer_sizes=(64, 32), random_state=42, max_iter=200)
}

# 5. Validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # Evaluar con cross_validate (incluye tiempos automáticamente)
    cvres = cross_validate(
        pipe, X_all, y, cv=cv,
        scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'},
        n_jobs=-1
    )

    # Predicciones OOF para Spearman
    y_pred_oof = cross_val_predict(pipe, X_all, y, cv=cv, n_jobs=-1)
    sp, _ = spearmanr(y, y_pred_oof)

    rows.append({
        "Model": name,
        "MAE_mean": -cvres["test_mae"].mean(),
        "RMSE_mean": -cvres["test_rmse"].mean(),
        "R2_mean": cvres["test_r2"].mean(),
        "Spearman_OOF": sp,
        "Fit_Time_s": cvres["fit_time"].mean() # <--- Tiempo solicitado en la rúbrica
    })

# 6. Mostrar tabla final ordenada por el mejor MAE
res_df = pd.DataFrame(rows).sort_values("MAE_mean")
display(res_df)

Shape de X_all: (2080, 27)


,Model,MAE_mean,RMSE_mean,R2_mean,Spearman_OOF,Fit_Time_s
1,Ridge,0.011237,0.014944,0.975908,0.979152,0.023276
2,Lasso,0.011714,0.015515,0.974042,0.979020,0.026671
4,DecisionTree,0.013153,0.018608,0.962477,0.971014,0.052152
3,ElasticNet,0.014179,0.018670,0.962487,0.974776,0.021273
5,KNN,0.020864,0.031162,0.895278,0.921812,0.017381
7,MLP (Neural Net),0.051810,0.074784,0.392830,0.719638,0.423452
6,SVR (RBF),0.057654,0.063373,0.566968,0.929889,0.034202
0,Dummy (Baseline),0.069882,0.096716,-0.002265,-0.035509,0.020289


---
## Ajuste Fino de Hiperparámetros (Fine-Tuning)

Observando la tabla comparativa anterior, notamos una mejora drástica en las métricas (con un $R^2$ cercano a 0.97) gracias al aumento sustancial de datos. 

Los dos modelos que demostraron el mejor rendimiento equilibrado (menor error y menor costo computacional) fueron la **Regresión Ridge** y el **Árbol de Decisión (Decision Tree)**. A continuación, exploraremos el espacio de búsqueda de estos algoritmos utilizando `GridSearchCV` para encontrar su configuración óptima y maximizar su capacidad de generalización en esta tarea específica.

---
## Elección del Modelo Individual Final

Tras evaluar 8 arquitecturas de aprendizaje automático individuales y optimizar las dos de mejor rendimiento, **se elige el modelo de Regresión Ridge como el modelo individual final de esta etapa.**

La justificación de esta elección se fundamenta en los siguientes criterios técnicos y de negocio:

1. **Rendimiento y Métricas Principales:** Ridge Regressor logró el mejor desempeño general, minimizando el Error Absoluto Medio (MAE) y maximizando el $R^2$. Su alta correlación de Spearman asegura que el modelo es altamente preciso para **rankear** la afinidad de los péptidos (`i_ptm`), lo cual es el objetivo core del *screening* in silico.
2. **Interpretabilidad (El factor caja blanca):** En la investigación biológica y farmacéutica, la interpretabilidad es crucial. Mientras que redes neuronales (MLP) o SVMs actúan como "cajas negras", la regresión Ridge nos permite extraer los coeficientes del modelo. Esto significa que los investigadores pueden validar empíricamente qué aminoácidos o propiedades estructurales están impulsando a que un fármaco sea un buen agonista de PD-L1.
3. **Complejidad y Requisitos Computacionales:** El tiempo de entrenamiento (`Fit_Time_s`) de Ridge es del orden de milisegundos. En un entorno de producción donde se deban inferir millones de mutaciones peptídicas para encontrar el candidato ideal, la complejidad lineal de Ridge garantiza una escalabilidad y latencia inmejorables comparado con algoritmos basados en distancias (KNN) o redes profundas.
4. **Trade-offs (Compensaciones):** Aunque un Árbol de Decisión profundo podría capturar ciertas no-linealidades, tiende a ser más inestable frente a ligeras variaciones en los datos. Ridge, por su naturaleza de regularización L2, penaliza pesos extremos, ofreciendo un *trade-off* excelente al ser un modelo robusto, generalizable y altamente confiable como filtro primario de candidatos.

---
## Conclusión General

En este cuarto avance, hemos dado un paso crucial hacia la consolidación de una herramienta predictiva confiable para el diseño de fármacos agonistas de PD-L1. 

La decisión estratégica de expandir nuestro conjunto de datos a 2080 registros demostró ser un punto de inflexión para el proyecto: logramos mitigar por completo los problemas de alta varianza y sobreajuste (overfitting) observados en las etapas iniciales, alcanzando niveles de precisión sobresalientes (con valores de $R^2$ superiores a 0.97).

**Principales hallazgos de esta etapa:**
* **Viabilidad de algoritmos individuales:** Comprobamos que, con un volumen de datos adecuado, no es estrictamente necesario recurrir a algoritmos de caja negra o ensambles complejos para obtener predicciones de alta calidad. Modelos lineales regularizados como **Ridge** y **Lasso** capturan de manera excelente la relación entre las características de la secuencia peptídica y la métrica de interacción `i_ptm`.
* **Impacto en el negocio (Drug Discovery):** La selección de la **Regresión Ridge** como nuestro modelo final nos brinda una ventaja competitiva enorme. Nos permite rankear millones de secuencias candidatas *in silico* en cuestión de segundos y con alta confiabilidad (Spearman > 0.97). Además, su transparencia nos permite interpretar qué factores físicos o químicos son los verdaderos impulsores de una buena unión proteína-proteína.

**Próximos pasos:**
Con una línea base individual sólida y optimizada, el equipo se encuentra en una posición ideal para las siguientes fases del proyecto. 

El modelo seleccionado servirá como un filtro primario altamente eficiente, reduciendo drásticamente los costos y tiempos asociados a la experimentación en laboratorio, y acercándonos al objetivo final de proponer secuencias de entrenamiento más utiles y eficazes en el entrenamiento de hormonas 